In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd().parent))
from src.models.train_model import generate_tsp_dataset

c:\Users\berna\anaconda3\envs\siwy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
X, y, _ = generate_tsp_dataset(
    n_cities=7,
    random_state=42,
    sample_size=360,
)

X_context, X_probe, y_context, y_probe = train_test_split(
    X, y, train_size=180, random_state=42,
)

X_probe_train, X_probe_test, y_probe_train, y_probe_test = (
    train_test_split(
        X_probe, y_probe, test_size=0.5, random_state=42,
    )
)

print("Kontekst TabPFN:", len(X_context))
print("Uczenie sond:", len(X_probe_train))
print("Ocena sond:", len(X_probe_test))

Kontekst TabPFN: 180
Uczenie sond: 90
Ocena sond: 90


In [4]:
X, y, _ = generate_tsp_dataset(
    n_cities=7,
    random_state=42,
    sample_size=360,
)

X_context, X_probe, y_context, y_probe = train_test_split(
    X, y, train_size=180, random_state=42,
)

X_probe_train, X_probe_test, y_probe_train, y_probe_test = (
    train_test_split(
        X_probe, y_probe, test_size=0.5, random_state=42,
    )
)

print("Kontekst TabPFN:", len(X_context))
print("Uczenie sond:", len(X_probe_train))
print("Ocena sond:", len(X_probe_test))

Kontekst TabPFN: 180
Uczenie sond: 90
Ocena sond: 90


In [5]:
from importlib.metadata import version
from tabpfn import TabPFNRegressor

print("Wersja TabPFN:", version("tabpfn"))

model = TabPFNRegressor(
    n_estimators=1,
    categorical_features_indices=list(range(X_context.shape[1])),
    random_state=42,
)

model.fit(X_context, y_context)

Wersja TabPFN: 10.0.0


,n_estimators,1
,categorical_features_indices,"[0, 1, ...]"
,random_state,42
,auto_scale_n_estimators,True
,group_col,None
,softmax_temperature,'auto'
,average_before_softmax,False
,model_path,'auto'
,device,'auto'
,ignore_pretraining_limits,False
,inference_precision,'auto'


In [6]:
if hasattr(model, "models_"):
    network = model.models_[0]
elif hasattr(model, "model_"):
    network = model.model_
else:
    raise RuntimeError("Nie znaleziono wewnętrznej sieci TabPFN.")

print("Typ sieci:", type(network))

for name, module in network.named_modules():
    print(name, type(module).__name__)

Typ sieci: <class 'tabpfn.architectures.tabpfn_v3_5.TabPFNV3p5'>
 TabPFNV3p5
x_embed FourierPlusMetadataFeatureGroupEmbedder
x_embed.fourier FourierFeatureGroupEmbedder
x_embed.fourier.in_linear Linear
x_embed.metadata_linear Linear
x_embed.layernorm LayerNorm
col_y_encoder ModuleDict
col_y_encoder.multiclass TrainableOrthogonalEmbedding
col_y_encoder.multiclass.embedding Embedding
col_y_encoder.regression Linear
col_y_layernorm LayerNorm
feature_distribution_embedder FeatureDistributionEmbedder
feature_distribution_embedder.layers ModuleList
feature_distribution_embedder.layers.0 InducedSelfAttentionBlock
feature_distribution_embedder.layers.0.cross_attn_block1 CrossAttentionBlock
feature_distribution_embedder.layers.0.cross_attn_block1.attn CrossAttention
feature_distribution_embedder.layers.0.cross_attn_block1.attn.softmax_scaling_layer SoftmaxScalingMLP
feature_distribution_embedder.layers.0.cross_attn_block1.attn.softmax_scaling_layer.base_mlp Sequential
feature_distribution_embed

In [7]:
import torch

def show_shape(module, inputs, output):
    if isinstance(output, torch.Tensor):
        print("Wyjście bloku:", tuple(output.shape))
    elif isinstance(output, (tuple, list)):
        print("Wyjścia bloku:", [
            tuple(x.shape) if isinstance(x, torch.Tensor)
            else type(x).__name__
            for x in output
        ])
    else:
        print("Typ wyjścia:", type(output))

handle = network.icl_blocks[0].register_forward_hook(show_shape)

try:
    predictions = model.predict(X_probe_train[:3])
finally:
    handle.remove()

Wyjścia bloku: [(1, 183, 1024), 'NoneType']


In [8]:
activations = []

def save_activations(module, inputs, output):
    hidden = output[0]
    activations.append(hidden[0, -3:, :].detach().cpu().numpy())

handle = network.icl_blocks[0].register_forward_hook(save_activations)

try:
    model.predict(X_probe_train[:3])
finally:
    handle.remove()

print("Liczba wywołań bloku:", len(activations))
print("Kształt reprezentacji:", activations[0].shape)

Liczba wywołań bloku: 1
Kształt reprezentacji: (3, 1024)


In [ ]:
def extract_representations(model, network, X):
    representations = {}
    call_counts = {}
    handles = []
    n_samples = len(X)

    def make_hook(layer_index):
        def save_activations(module, inputs, output):
            hidden = output[0] if isinstance(output, (tuple, list)) else output

            call_counts[layer_index] = call_counts.get(layer_index, 0) + 1

            representations[layer_index] = (
                hidden[0, -n_samples:, :]
                .detach()
                .cpu()
                .float()
                .numpy()
                .copy()
            )

        return save_activations

    try:
        for layer_index, block in enumerate(network.icl_blocks):
            handle = block.register_forward_hook(make_hook(layer_index))
            handles.append(handle)

        predictions = model.predict(X)

    finally:
        for handle in handles:
            handle.remove()

    for layer_index in range(len(network.icl_blocks)):
        count = call_counts.get(layer_index, 0)

    return representations, predictions

In [ ]:
train_representations, train_predictions = extract_representations(
    model,
    network,
    X_probe_train,
)

test_representations, test_predictions = extract_representations(
    model,
    network,
    X_probe_test,
)


Blok 01: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 02: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 03: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 04: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 05: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 06: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 07: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 08: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 09: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 10: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 11: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 12: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 13: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 14: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 15: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 16: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 17: uczenie sondy (90, 1024), test sondy (90, 1024)
Blok 18: uczenie sondy (90, 102

In [12]:
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [15]:
def fit_probe(X_train, y_train):
    pipeline = make_pipeline(
        StandardScaler(),
        Ridge(solver="svd"),
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid={
            "ridge__alpha": np.logspace(-3, 5, 9),
        },
        scoring="neg_mean_absolute_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=1,
    )
    search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_["ridge__alpha"]


probes = {}
results = []

for layer_index in sorted(train_representations):
    probe, alpha = fit_probe(
        train_representations[layer_index],
        y_probe_train,
    )

    predictions = probe.predict(
        test_representations[layer_index]
    )

    probes[layer_index] = probe

    results.append({
        "block": layer_index + 1,
        "mae": mean_absolute_error(y_probe_test, predictions),
        "r2": r2_score(y_probe_test, predictions),
        "alpha": alpha,
    })

results_df = pd.DataFrame(results)

print("\nJakość sond na trasach testowych:")
print(
    results_df.to_string(
        index=False,
        formatters={
            "mae": "{:.4f}".format,
            "r2": "{:.4f}".format,
            "alpha": "{:.3g}".format,
        },
    )
)


Jakość sond na trasach testowych:
 block    mae     r2 alpha
     1 0.0092 0.9992     1
     2 0.0091 0.9992     1
     3 0.0090 0.9992     1
     4 0.0101 0.9990    10
     5 0.0096 0.9990 0.001
     6 0.0101 0.9990    10
     7 0.0114 0.9986    10
     8 0.0125 0.9984    10
     9 0.0136 0.9983    10
    10 0.0133 0.9983    10
    11 0.0148 0.9978    10
    12 0.0143 0.9980    10
    13 0.0141 0.9980    10
    14 0.0136 0.9983    10
    15 0.0140 0.9982    10
    16 0.0142 0.9981    10
    17 0.0108 0.9989     1
    18 0.0111 0.9987     1
    19 0.0107 0.9986 0.001
    20 0.0109 0.9984 0.001
    21 0.0115 0.9983    10
    22 0.0137 0.9969    10
    23 0.0144 0.9948    10
    24 0.0175 0.9905 0.001
